In [19]:
# global so both the callback and the main loop can access the last clicked box
last_click_box = None

def show_hsv_on_click(event, x, y, flags, param):
    global last_click_box

    if event == cv.EVENT_LBUTTONDOWN:
        hsv_frame = param
        h, s, v = [int(val) for val in hsv_frame[y, x]]
        print("HSV at click:", (h, s, v))

        # Build a small tolerance range around the clicked pixel's HSV,
        # so we catch the whole object, not just that one exact pixel.
        tol_h, tol_s, tol_v = 10, 40, 40
        lower = np.array([max(h - tol_h, 0), max(s - tol_s, 0), max(v - tol_v, 0)])
        upper = np.array([min(h + tol_h, 179), min(s + tol_s, 255), min(v + tol_v, 255)])

        mask = cv.inRange(hsv_frame, lower, upper)
        contours, _ = cv.findContours(mask, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)

        # Find whichever contour actually contains the clicked point
        # (there could be many small blobs matching similar HSV elsewhere in frame)
        for contour in contours:
            if cv.pointPolygonTest(contour, (x, y), False) >= 0:
                last_click_box = cv.boundingRect(contour)  # (bx, by, bw, bh)
                break

In [20]:
def color_identification(frame):
    # Convert the frame to HSV color space
    hsv_frame = cv.cvtColor(frame, cv.COLOR_BGR2HSV)

    # HSV (Hue, Saturation, Value) 
    # separates color information (hue) from intensity information (value), 
    # making it easier to detect specific colors under varying lighting conditions.
    
    # Define color ranges for detection (example: red color)
    # Why do we have multiple lower and upper bounds for red? 
    # Because red wraps around the hue value in HSV color space, 
    # so we need to account for both ends of the spectrum.
    
    lower_red = np.array([0, 120, 70])
    upper_red = np.array([10, 255, 255])
    mask1 = cv.inRange(hsv_frame, lower_red, upper_red)

    lower_red2 = np.array([170, 120, 70])
    upper_red2 = np.array([180, 255, 255])
    mask2 = cv.inRange(hsv_frame, lower_red2, upper_red2)

    lower_black = np.array([0, 0, 0])
    upper_black = np.array([180, 30, 60])
    mask3 = cv.inRange(hsv_frame, lower_black, upper_black)

    # Combine masks
    # combine masks because red color can be detected in two different ranges in HSV color space,
    #  and we want to capture all red objects in the frame under both ranges. 
    # By combining the masks, we ensure that we don't miss any red objects 
    # that might fall into either range.
    
    black_mask = mask3 
    red_mask = mask1 #| mask2  
    # | is a bitwise or operator, that compares each binary value of the masks.
    # info from openCV basically organizes the pixels in the below format: 
    
    # Each mask is a binary/grayscale image:
    # 255 = pixel matches the specified HSV range
    # 0   = pixel does not match the specified HSV range.
    #
    # If either mask contains 255 at a particular pixel, the combined
    # mask will contain 255 at that pixel. If both contain 0, the
    # combined mask contains 0.
    #
    # This gives us one mask containing pixels that match either
    # of the two red HSV ranges.
    
    # basically says a mask either has the color red -- blue, green, whatever -- (represented by number 255), or it doesn't (represented by rgb 0).
    # The bitwise OR operator converts each bit/pixel of the two masks into a binary value
    # and accordingly combines the BINARY VALUES of the two masks, to get the most accurate representation of red
    # and merges them into a single mask, taking the most accurate represntation of the color red
    # Basically says to "Keep a pixel if either mask says it's red"


    # Find contours of the detected color
    # contours are curves that join all the continuous points 
    # along a boundary with the same color or intensity
    # significant because they help in identifying the shape and size of the detected object
    
    red_contours, _ = cv.findContours(red_mask, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)
    #green_contours, _ = cv.findContours(green_mask, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)
    black_contours, _ = cv.findContours(black_mask, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)


    for contour in red_contours:
        area = cv.contourArea(contour)
        if area > 2000:  # Filter out small areas; area identified needs to be at least 2000 pixels to be considered a valid detection
            x, y, w, h = cv.boundingRect(contour)
            cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv.putText(frame, 'Red Object', (x, y - 10), cv.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    for contour in black_contours:
        area = cv.contourArea(contour)
        if area > 5000:  # Filter out small areas; area identified needs to be at least 5000 pixels to be considered a valid detection
            x, y, w, h = cv.boundingRect(contour)
            cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv.putText(frame, 'Black Object', (x, y - 10), cv.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    return frame

**Initialze the live feed and display frames. Create a basic command to break out of it**

In [ ]:
import cv2 as cv
import numpy as np

#init camera
cap = cv.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Can't receive frame.")
        break

    #loads the color filter for the frame
    frame = cv.flip(frame, 1)
    # 1 flips the frame horizontally, creating a mirror effect
    # 0 flips the frame vertically creating an upside-down un-mirrored effect
    # -1 flips the frame vertically and horizontally, creating a mirrored upside-down effect


    color_detection = color_identification(frame)
    #Displays frames
    cv.imshow('Feed', color_detection)



    #renders each and every frame properly from .imshow, and produces the livestream; 
    # 1 is a param for millimtere refresh rate
    # press q to break
    # OxFF is a hex number that is standardized across all systems and standardizes that the 
    # key pressed (in this case, q) is the same across all systems
    # 2nd part of or condition basically says if the life feed window isn't visible 
    # (aka closed with mouse), then break the loop and stop the program
    
    if cv.waitKey(1) & (0xFF == ord('q') or cv.getWindowProperty('Feed', cv.WND_PROP_VISIBLE) < 1):
        break

cap.release()          # Turns off your webcam light and frees up the camera
cv.destroyAllWindows() # Completely closes the 'Feed' window on your screen

KeyboardInterrupt: 

: 

## Breaking it down piece by piece:

* cap.read()
This command reaches out to your webcam and says, "Give me the exact image you are looking at right this millisecond."
* ret, frame =
The camera gives you back two answers at the exact same time, which are saved into two separate variables:
* frame: This is the actual, physical picture (a matrix of pixel colors) that you display on the screen.
   * ret: Short for "Return." This is just a simple True or False value. If the camera successfully grabbed the picture, ret is True. If something went wrong, ret is False.
* if not ret:
This checks the success signal. It translates to: "If ret is NOT True (meaning it is False), execute the code below."
* print(...) and break
If the camera failed (for example, if you unplugged the USB cord mid-stream or another app stole the camera access), it alerts you in the terminal and breaks out of the loop so your code doesn't crash trying to display a blank image.
* OpenCV windows cannot render images on their own. When you call cv.imshow(), OpenCV doesn't actually draw the picture immediately; it just puts the frame into a "waiting room." cv.waitKey() is the only command that forces the operating system to actually paint the pixels onto your monitor.

# FINAL NOTES FOR: color_analysis.ipynb -- OpenCV Color Segmentation & Detection

**Pipeline:** `Image → HSV → color mask(s) → combine → contours → filter by area → bounding box → draw`

---

## 1. BGR vs. HSV

OpenCV loads images as **BGR** (Blue, Green, Red) instead of the usual RGB order.

For color filtering, **HSV** is easier to work with:

| Channel | Meaning | Range in OpenCV |
|---|---|---|
| **H**ue | the base color (red, green, blue...) | `0 – 179` |
| **S**aturation | how pure/vivid the color is (gray → vivid) | `0 – 255` |
| **V**alue | brightness (dark → bright) | `0 – 255` |

> Note: most online HSV charts show hue as `0–360°`. OpenCV halves this to `0–179`.

**Why HSV helps:** instead of reasoning about 3 mixed color channels (BGR), you can ask directly:
- *"What color is it?"* → H
- *"How strong is the color?"* → S
- *"How bright is it?"* → V

Convert an image with:
```python
hsv_frame = cv.cvtColor(frame, cv.COLOR_BGR2HSV)
```
(Keep the original `frame` too — you still need it later for drawing.)

---

## 2. Defining a Color Range

```python
lower_red = np.array([0, 120, 70])
upper_red = np.array([10, 255, 255])
```

A pixel is selected only if **all three** conditions hold:
- `0 ≤ H ≤ 10`
- `120 ≤ S ≤ 255`
- `70 ≤ V ≤ 255`

### Why red needs *two* ranges

Hue is a circular scale (`0 → 179 → wraps to 0`), and red sits at **both ends**:

```
0 ───────────────────── 179
↑                          ↑
RED                      RED
```

So red detection typically uses two ranges:
```python
lower_red,  upper_red   = [0,120,70],   [10,255,255]
lower_red2, upper_red2  = [170,120,70], [180,255,255]
```

---

## 3. Building the Mask

```python
mask1 = cv.inRange(hsv_frame, lower_red,  upper_red)
mask2 = cv.inRange(hsv_frame, lower_red2, upper_red2)
red_mask = mask1 | mask2   # bitwise OR — combine both ranges
```

- `cv.inRange()` checks every pixel against the HSV range → outputs **255** (matched) or **0** (didn't match).
- **Important:** `255` doesn't mean "this pixel is red" — it means *"this pixel passed the filter I defined."*
- `|` is **bitwise OR** (not Python's `or`). A pixel survives in `red_mask` if it matched *either* mask:

| mask1 | mask2 | result |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 255 | 255 |
| 255 | 0 | 255 |
| 255 | 255 | 255 |

The result, `red_mask`, is a **binary mask** — a black-and-white image marking which pixels are candidates.

---

## 4. From Pixels to Regions: Contours

A mask only tells you *which pixels* matched. It doesn't tell you *where the objects are*, *how big they are*, or *how many there are*. That's what **contours** give you.

```python
contours, _ = cv.findContours(
    red_mask,
    cv.RETR_TREE,
    cv.CHAIN_APPROX_SIMPLE
)
```

| Part | Meaning |
|---|---|
| `red_mask` | the binary image being analyzed (not the original color image) |
| `cv.RETR_TREE` | keep hierarchy info (outer/inner contours, e.g. shapes with holes). Use `cv.RETR_EXTERNAL` if you only want outermost shapes |
| `cv.CHAIN_APPROX_SIMPLE` | compress redundant points along straight edges (keeps the shape, drops unneeded detail) |
| `contours, _` | `findContours` returns `(contours, hierarchy)`; the `_` throws away hierarchy since we're not using it |

`contours` is a list — one entry per detected connected region:
```
contours = [contour_0, contour_1, contour_2, ...]
```

---

## 5. Filtering & Measuring Contours

```python
for contour in contours:
    area = cv.contourArea(contour)

    if area > 500:
        x, y, w, h = cv.boundingRect(contour)
        ...
```

- **`cv.contourArea()`** — approximate area enclosed by the contour.
- **`if area > 500`** — drop tiny contours (noise, reflections, compression artifacts, stray pixels). `500` is *not* universal — tune it per image resolution / camera distance / lighting.
- **`cv.boundingRect()`** — returns the smallest upright rectangle containing the contour:
  - `x, y` = top-left corner
  - `w, h` = width, height
  - Image coordinates: `x` grows **right**, `y` grows **down** (origin is top-left, unlike a math graph).

---

## 6. Drawing the Result

```python
cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
cv.putText(frame, 'Red Object', (x, y - 10),
           cv.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
```

- `(x, y)` → top-left corner, `(x + w, y + h)` → bottom-right corner.
- `(0, 255, 0)` is **BGR**, so this is green — don't confuse this with mask values (`0`/`255` there means selected/rejected, not color).
- Label is drawn slightly above the box (`y - 10`) via `cv.putText`.

---

## 7. Full Function

```python
def color_identification(frame):
    hsv_frame = cv.cvtColor(frame, cv.COLOR_BGR2HSV)

    # Two hue ranges because red wraps around the hue scale
    lower_red,  upper_red  = np.array([0,120,70]),   np.array([10,255,255])
    lower_red2, upper_red2 = np.array([170,120,70]), np.array([180,255,255])

    mask1 = cv.inRange(hsv_frame, lower_red,  upper_red)
    mask2 = cv.inRange(hsv_frame, lower_red2, upper_red2)
    red_mask = mask1 | mask2

    contours, _ = cv.findContours(red_mask, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        area = cv.contourArea(contour)
        if area > 500:
            x, y, w, h = cv.boundingRect(contour)
            cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv.putText(frame, 'Red Object', (x, y - 10),
                       cv.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    return frame
```

---

## 8. Mental Model

```
Pixels → HSV values → color filter → mask (0/255)
      → connected regions → contours
      → area filter → bounding box → drawn detection
```

**HSV filtering** decides *which pixels count*. **The mask** records that decision. **`findContours()`** finds the boundaries of connected selected pixels. From there you can measure size (`contourArea`) and location (`boundingRect`).

> This is **color segmentation**, not true object recognition — it's finding "pixels matching my red definition, in a big enough connected blob," not "this is a ball." It breaks down when lighting shifts, another object shares the color, or the object is partly occluded.

---

## 9. Where This Goes Next

- **Contour moments / centroid** → track a single point per object instead of a whole box
- **Object tracking across frames** → follow position over time → estimate velocity
- **Morphological operations** (erode/dilate) → clean up noisy masks
- **`cv.RETR_EXTERNAL`** → simpler hierarchy when you don't need nested contours